In [5]:
import os
from ratelimit import limits, sleep_and_retry
import requests
import json
from urllib3.util import Retry

In [6]:
EPOCH_TIME_JULY2026 = 1782867600
RANKED_SOLO = 420
ONE_SECOND = 1
TWO_MINUTES = 120
NUM_CHAMPIONS_PER_GAME = 10
NUM_GAMES_PER_PLAYER = 1
CURRENT_PATCH = '16.14.1' # Update this with the current patch version

platforms = ['OC1', 'JP1', 'KR', 'BR1', 'LA1', 'LA2', 'NA1', 'TR1', 'RU', 'EUN1', 'EUW1', 'ME1', 'SG2', 'TW2', 'VN2']
regions = {'OC1':'sea', 'SG2': 'sea', 'TW2': 'sea', 'VN2': 'sea', 'JP1': 'asia', 'KR': 'asia', 'BR1': 'americas', 'LA1': 'americas', 'LA2': 'americas', 'NA1': 'americas', 'TR1': 'europe', 'RU': 'europe', 'EUN1': 'europe', 'EUW1': 'europe', 'ME1': 'europe'}

champion_names_url = 'https://ddragon.leagueoflegends.com/cdn/{version}/data/en_US/champion.json'
master_division_url = 'https://{platform}.api.riotgames.com/lol/league/v4/masterleagues/by-queue/RANKED_SOLO_5x5'
matches_by_player_url = 'https://{region}.api.riotgames.com/lol/match/v5/matches/by-puuid/{puuid}/ids?startTime={start_time}&queue={queue}&type=ranked&start=0&count={count}'
match_data_from_matchid = 'https://{region}.api.riotgames.com/lol/match/v5/matches/{matchId}'
api_key = os.getenv("RIOT_API_KEY")

headers = {
    'X-Riot-Token': api_key
}


In [7]:
session = requests.Session()
retries = Retry(total=10,
                backoff_factor=2,
                status_forcelist=[429, 500, 502, 503, 504])
session.mount('https://', requests.adapters.HTTPAdapter(max_retries=retries))

In [8]:
@sleep_and_retry
@limits(calls=90, period=TWO_MINUTES)
@limits(calls=18, period=ONE_SECOND)
def call_api(url, headers=None):
    response = session.get(url, headers=headers)

    if response.status_code >= 400:
        print(f'Status: {response.status_code} Url: {url}')
        return None
    
    return response

In [ ]:
for platform in platforms[:1]:
    player_data = call_api(master_division_url.format(platform=platform), headers)

    data = player_data.json()['entries']
    player_id = [player['puuid'] for player in data] # collects all the player puuids from the master division

    match_data = []
    for puuid in player_id:
        url = matches_by_player_url.format(region=regions[platform],
                                           puuid=puuid,
                                           start_time=EPOCH_TIME_JULY2026,
                                           queue=RANKED_SOLO,
                                           count=NUM_GAMES_PER_PLAYER)
        response = call_api(url, headers)
        
        if not response:
            continue
        match_data.extend(response.json())
        print(f'{len(player_data) * NUM_GAMES_PER_PLAYER - len(match_data)} matches left to go')

1794
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179


KeyboardInterrupt: 

In [ ]:
match_data = list(set(match_data)) # removes duplicates

In [ ]:
champ_data = []
for match_id in match_data[:5]: # only collects data from the first 5 matches for now
    game_data = call_api(match_data_from_matchid.format(matchId=match_id), headers)
    current_champs = [game_data.json()['info']['participants'][i]['championName'] for i in range(NUM_CHAMPIONS_PER_GAME)] # first 5 participants are team 1, next 5 are team 2
    champ_data.append(current_champs)

In [ ]:
all_champion_names = call_api(champion_names_url.format(version=CURRENT_PATCH))

session.close()

In [80]:
with open('champ_names.json', 'w') as f:
    json.dump(list(all_champion_names.json()['data'].keys()), f)

In [63]:
with open('champ_data.json', 'w') as f:
    json.dump(champ_data, f)